In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import files
import os, zipfile, glob
if not os.path.isdir('spin360'):
    up = files.upload()  # choose spin360.zip
    z = next(iter(up))
    with zipfile.ZipFile(z) as zf: zf.extractall('.')
# land in the package root (the folder that contains requirements.txt)
root = next(p for p in ['spin360', '.'] if os.path.exists(os.path.join(p,'requirements.txt'))
            ) if glob.glob('**/requirements.txt', recursive=True) else 'spin360'
%cd $root
print('cwd:', os.getcwd())

In [ ]:
# python deps
!pip -q install fastapi 'uvicorn[standard]' python-multipart pydantic SQLAlchemy \
    numpy Pillow opencv-python-headless trimesh requests 2>/dev/null
print('python deps ok')

In [ ]:
# headless Blender 4.2 LTS for the GPU render backend
import os, subprocess
BL='blender-4.2.0-linux-x64'
if not os.path.isdir(f'/opt/{BL}'):
    !wget -q https://download.blender.org/release/Blender4.2/{BL}.tar.xz -O /tmp/bl.tar.xz
    !mkdir -p /opt && tar -xf /tmp/bl.tar.xz -C /opt
os.environ['SPIN360_BLENDER_BIN']=f'/opt/{BL}/blender'
print(subprocess.run([os.environ['SPIN360_BLENDER_BIN'],'--version'],
                     capture_output=True,text=True).stdout.strip())

In [ ]:
import os, time, subprocess, threading
os.environ.update(
    SPIN360_DATA_DIR='/content/spin360_data',
    SPIN360_DB_URL='sqlite:////content/spin360_data/spin360.db',
    SPIN360_INLINE='1',            # in-process worker; UI polls live stage progress
    SPIN360_RECON_PROVIDER='mock', # swap to local_trellis in step 5 for real 3D
    SPIN360_RENDER='cpu',          # fast + reliable for the mock; step 5 switches to Blender GPU
    SPIN360_BLENDER_ENGINE='CYCLES',
    SPIN360_BLENDER_SAMPLES='16',
)
def _serve():
    subprocess.run(['uvicorn','spin360.api:app','--host','0.0.0.0','--port','8000'])
threading.Thread(target=_serve, daemon=True).start()
time.sleep(6)
from google.colab.output import eval_js
print('Open your Spin360 UI:')
print(eval_js('google.colab.kernel.proxyPort(8000)'))

In [ ]:
from google.colab import files
from IPython.display import Video, display
print('Upload the FRONT photo:'); f=files.upload(); front=next(iter(f))
print('Upload the BACK photo:');  b=files.upload(); back=next(iter(b))
import subprocess, glob, os
env=dict(os.environ, SPIN360_DATA_DIR='/content/oneshot',
         SPIN360_DB_URL='sqlite:////content/oneshot/db.sqlite',
         SPIN360_RENDER='blender', SPIN360_RECON_PROVIDER=os.environ.get('SPIN360_RECON_PROVIDER','mock'))
subprocess.run(['python','scripts/run_local.py',front,back], env=env, check=True)
mp4=sorted(glob.glob('/content/oneshot/**/spin.mp4', recursive=True))[-1]
display(Video(mp4, embed=True, width=420))

In [ ]:
# TRELLIS install (see github.com/microsoft/TRELLIS for the authoritative script)
import os
if not os.path.isdir('TRELLIS'):
    !git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd TRELLIS
# their setup installs torch + the sparse/rasterization CUDA ops; ~5-10 min on T4
!. ./setup.sh --basic --xformers --flash-attn --diffoctreerast --spconv --mipgaussian --nvdiffrast 2>&1 | tail -5
%cd ..
import sys; sys.path.insert(0,'TRELLIS')
os.environ['SPIN360_RECON_PROVIDER']='local_trellis'
os.environ['SPIN360_RENDER']='blender'   # real geometry -> render on the T4 GPU
os.environ['ATTN_BACKEND']='flash-attn'; os.environ['SPCONV_ALGO']='native'
print('TRELLIS ready — re-run step 3 (UI) or step 4 (one-shot) for real 3D.')
print('Watch the Blender log for: [spin360] Cycles GPU: OPTIX -> [Tesla T4].')
print('If it says WARNING: no GPU, keep SPIN360_RENDER=cpu instead.')